# 第 37 课：时间戳、说话人分段与 Diarization

ASR 回答“说了什么”，diarization 回答“谁在什么时候说”。教学实验用 FSDD 多说话人开源录音建立简单声纹原型。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 后处理与语义 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 36 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | token 时间戳、speaker embedding、DER |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：token 时间戳、speaker embedding、DER。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：raw ASR 与 normalized text 的证据边界；置信度；N-best；会话状态隔离。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](36_流式音频前端总管线与状态.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：带时间、说话人、候选与置信度的可追溯识别结果
  ↓ 本课要学会的变换、状态或判断
输出：保留原证据、可校准、可拒绝或澄清的文本/语义结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from collections import defaultdict
files=sorted((ROOT/"data"/"fsdd_multispeaker").glob("*.wav"));print(len(files),files[:3])

## 1. 教学版 speaker embedding

In [ ]:
def embedding(path):
    y,sr=sf.read(path);m=librosa.feature.mfcc(y=y.astype(np.float32),sr=sr,n_mfcc=13,n_fft=256,hop_length=80)
    e=np.concatenate([m.mean(1),m.std(1)]);return e/(np.linalg.norm(e)+1e-8)
groups=defaultdict(list)
for p in files:groups[p.stem.split("_")[1]].append((p,embedding(p)))
prototypes={s:np.mean([e for _,e in items],axis=0) for s,items in groups.items()}
for s in prototypes:prototypes[s]/=np.linalg.norm(prototypes[s])+1e-8
for s,items in groups.items():
    p,e=items[0];scores={name:float(e@proto) for name,proto in prototypes.items()};print(p.name,"->",max(scores,key=scores.get),scores)

这不是生产声纹模型：数据太少，而且 enrollment 与测试复用。它只展示 embedding→相似度→聚类/匹配的接口。

## 2. 时间戳的来源

- 帧时间：由采样计数、window/hop 得到；
- CTC spike：可给 token 粗时间；
- forced alignment：在已知文本条件下寻找更精确对齐；
- segment 时间：由 VAD/diarization 边界得到。

经过卷积下采样、右上下文和重采样后，必须维护从 encoder step 回到原始样本的映射。

## 3. Overlap speech

普通 diarization 常假设一帧一个 speaker；两人重叠时这个假设失效，需要 overlap detection 或多输出源分离。DER 也通常拆成 miss、false alarm、speaker confusion。

## 本课测试

1. speaker recognition 与 diarization 有何不同？
2. CTC spike 时间是否等于真实音素边界？
3. 为什么不能用同一录音同时 enrollment 和测试并宣称准确？
4. 两人同时说话为何困难？
5. DER 包含哪些主要错误？

<details><summary>展开参考答案</summary>

1. 前者识别身份，后者把时间轴按说话人切分/聚类。2. 不等于，只是模型输出峰。3. 会数据泄漏。4. 单标签帧假设失效。5. 漏检、误检和说话人混淆。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 37 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `token 时间戳`、`speaker embedding`、`DER`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**enrollment 和 test 使用同一录音**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**做 speaker-disjoint 验证并报告混淆**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接说话人片段与最终文本结构**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：token 时间戳、speaker embedding、DER。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 token 时间戳、speaker embedding、DER。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
